In [1]:
import numpy as np
import pandas as pd
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from matplotlib import pyplot
import imblearn
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_curve, auc, RocCurveDisplay
from sklearn import datasets
from sklearn.preprocessing import LabelBinarizer

In [6]:
df = pd.read_excel("/content/perfect_final_data_for_multilevel_analysis (1).xlsx")
df

,V001,V002,V012,V013,V024,V025,V106,V113,V130,V157,...,Number_of_children,Living_children,Age_of_husband,profession_of_husband,Birth_Interval,Drinking_water,Wealth_Status,Religion_status,BMI,Respondent_Age_first_birth
0,469,57,48,7,6,2,0,21,1,0,...,2,2,2,1,4,2,0,1,4,1
1,438,26,32,4,6,1,2,21,1,0,...,1,1,1,2,1,2,2,1,4,0
2,296,96,39,5,4,1,2,21,1,0,...,2,2,2,4,1,2,2,1,4,1
3,438,4,48,7,6,1,0,21,1,0,...,2,2,2,2,1,2,2,1,4,0
4,548,158,32,4,7,2,3,12,1,0,...,1,1,1,1,2,1,0,1,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8497,417,132,41,6,5,2,1,21,1,0,...,2,1,2,1,3,2,0,1,1,0
8498,190,11,43,6,3,1,0,11,1,0,...,1,1,1,3,3,1,1,1,1,1
8499,123,123,40,6,2,2,0,21,1,0,...,1,1,2,2,1,2,0,1,1,1
8500,581,64,30,4,7,2,1,21,2,0,...,2,2,1,2,2,2,0,0,1,1


In [7]:
df.drop(columns=['V001', 'V002', 'V012', 'V113', 'V130', 'V157', 'V158',
                 'V159', 'V190', 'V201', 'V212', 'V218', 'V221', 'V364', 'V445','V447', 'V501',
                 'V502', 'V701', 'V705', 'V730'], inplace=True)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8502 entries, 0 to 8501
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   V013                        8502 non-null   int64  
 1   V024                        8502 non-null   int64  
 2   V025                        8502 non-null   int64  
 3   V106                        8502 non-null   int64  
 4   V404                        8502 non-null   int64  
 5   V714                        8502 non-null   int64  
 6   BMI_fixed                   8502 non-null   float64
 7   Media_acccess               8502 non-null   int64  
 8   husband_education           8502 non-null   int64  
 9   contraceptive_use           8502 non-null   int64  
 10  Number_of_children          8502 non-null   int64  
 11  Living_children             8502 non-null   int64  
 12  Age_of_husband              8502 non-null   int64  
 13  profession_of_husband       8502 

In [10]:
df.drop(columns=['BMI_fixed'], inplace=True)

In [11]:
# Standardization (FT1)
from sklearn.preprocessing import StandardScaler  # import the standard scaler
FT1 = StandardScaler()  # create the scaler object
FT1_data = FT1.fit_transform(df)  # scale all columns to similar ranges
FT1_df = pd.DataFrame(FT1_data, columns=df.columns)  # turn scaled data into a DataFrame
print(FT1_df.head())  # show the first few scaled rows

       V013      V024      V025      V106     V404      V714  Media_acccess  \
0  1.654341  0.721957  0.726851 -1.787211 -0.55969 -0.691444      -1.182501   
1 -0.107701  0.721957 -1.375798  0.453920 -0.55969  1.446249       0.845666   
2  0.479646 -0.173524 -1.375798  0.453920 -0.55969 -0.691444       0.845666   
3  1.654341  0.721957 -1.375798 -1.787211 -0.55969 -0.691444       0.845666   
4 -0.107701  1.169697  0.726851  1.574486 -0.55969 -0.691444      -1.182501   

   husband_education  contraceptive_use  Number_of_children  Living_children  \
0          -1.394010          -1.469896            0.887196         1.375821   
1           0.570497          -1.469896           -0.738788        -0.698156   
2           1.552751           0.680320            0.887196         1.375821   
3           0.570497          -1.469896            0.887196         1.375821   
4          -0.411756          -1.469896           -0.738788        -0.698156   

   Age_of_husband  profession_of_husband  Bi

In [13]:

y = FT1_df.BMI  # use bmi1 as the target variable
x = FT1_df.drop('BMI', axis=1)  # use the other columns as features

In [14]:
x  # display the feature columns

,V013,V024,V025,V106,V404,V714,Media_acccess,husband_education,contraceptive_use,Number_of_children,Living_children,Age_of_husband,profession_of_husband,Birth_Interval,Drinking_water,Wealth_Status,Religion_status,Respondent_Age_first_birth
0,1.654341,0.721957,0.726851,-1.787211,-0.559690,-0.691444,-1.182501,-1.394010,-1.469896,0.887196,1.375821,1.467388,-1.147107,0.767338,0.309923,-1.174877,0.344506,1.042787
1,-0.107701,0.721957,-1.375798,0.453920,-0.559690,1.446249,0.845666,0.570497,-1.469896,-0.738788,-0.698156,-0.089160,-0.332277,-0.940343,0.309923,1.069042,0.344506,-0.958969
2,0.479646,-0.173524,-1.375798,0.453920,-0.559690,-0.691444,0.845666,1.552751,0.680320,0.887196,1.375821,1.467388,1.297384,-0.940343,0.309923,1.069042,0.344506,1.042787
3,1.654341,0.721957,-1.375798,-1.787211,-0.559690,-0.691444,0.845666,0.570497,-1.469896,0.887196,1.375821,1.467388,-0.332277,-0.940343,0.309923,1.069042,0.344506,-0.958969
4,-0.107701,1.169697,0.726851,1.574486,-0.559690,-0.691444,-1.182501,-0.411756,-1.469896,-0.738788,-0.698156,-0.089160,-1.147107,-0.371116,-1.122123,-1.174877,0.344506,1.042787
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8497,1.066994,0.274216,0.726851,-0.666646,-0.559690,-0.691444,-1.182501,-1.394010,0.680320,0.887196,-0.698156,1.467388,-1.147107,0.198111,0.309923,-1.174877,0.344506,-0.958969
8498,1.066994,-0.621265,-1.375798,-1.787211,-0.559690,-0.691444,0.845666,-1.394010,-1.469896,-0.738788,-0.698156,-0.089160,0.482554,0.198111,-1.122123,-0.052918,0.344506,1.042787
8499,1.066994,-1.069005,0.726851,-1.787211,-0.559690,-0.691444,-1.182501,-1.394010,0.680320,-0.738788,-0.698156,1.467388,-0.332277,-0.940343,0.309923,-1.174877,0.344506,1.042787
8500,-0.107701,1.169697,0.726851,-0.666646,1.786703,-0.691444,0.845666,-0.411756,0.680320,0.887196,1.375821,-0.089160,-0.332277,-0.371116,0.309923,-1.174877,-2.902709,1.042787


In [15]:
y = LabelEncoder().fit_transform(y)  # change target labels into numbers
oversample = SMOTE()  # create SMOTE to balance classes
x, y = oversample.fit_resample(x, y)  # make class counts more balanced
counter = Counter(y)  # count samples in each class
for k, v in counter.items():  # loop through each class count
  per = v / len(y) * 100  # calculate class percentage
  print('class=%d, count=%d, percentage=%.3f%%' % (k, v, per))  # print class distribution

class=3, count=4469, percentage=25.000%
class=2, count=4469, percentage=25.000%
class=1, count=4469, percentage=25.000%
class=0, count=4469, percentage=25.000%


In [16]:
from sklearn.model_selection import train_test_split  # import the train-test split tool
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=101)  # split data into training and testing sets
x_train.head()  # preview the training features
x_train.shape  # check the training data shape

(12513, 18)

In [17]:
x_test.head()  # preview the testing features
x_test.shape  # check the testing data shape

(5363, 18)

In [18]:
# logistic regression
from sklearn.linear_model import LogisticRegression  # import logistic regression
Lr = LogisticRegression()  # create the model
Lr.fit(x_train, y_train)  # train the model on the training data

# Generate predictions for both test and training data
predictions = Lr.predict(x_test)  # predict labels for the test set
train_predictions = Lr.predict(x_train)  # predict labels for the training set

from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools

# Train performance
print("=== Training Performance ===")  # print training results heading
print(classification_report(y_train, train_predictions, digits=4))  # show training metrics
print(confusion_matrix(y_train, train_predictions))  # show training confusion matrix
# test performance
print("=== Test Performance ===")  # print test results heading
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = Lr.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities for ROC work

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # measure agreement between true and predicted labels

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(Lr, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

=== Training Performance ===
              precision    recall  f1-score   support

           0     0.4514    0.6087    0.5184      3120
           1     0.3009    0.1915    0.2340      3113
           2     0.3516    0.2227    0.2727      3112
           3     0.4488    0.6168    0.5195      3168

    accuracy                         0.4109     12513
   macro avg     0.3882    0.4099    0.3861     12513
weighted avg     0.3885    0.4109    0.3868     12513

[[1899  600  274  347]
 [1239  596  525  753]
 [ 660  459  693 1300]
 [ 409  326  479 1954]]
=== Test Performance ===
              precision    recall  f1-score   support

           0     0.4485    0.6034    0.5145      1349
           1     0.3154    0.1947    0.2408      1356
           2     0.3785    0.2388    0.2928      1357
           3     0.4399    0.6272    0.5171      1301

    accuracy                         0.4136      5363
   macro avg     0.3956    0.4160    0.3913      5363
weighted avg     0.3950    0.4136    0

In [19]:
# Decision Tree
from sklearn.tree import DecisionTreeClassifier  # import decision tree model
dtree = DecisionTreeClassifier()  # create the model
dtree.fit(x_train, y_train)  # train the model
predictions = dtree.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = dtree.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(dtree, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.6799    0.7087    0.6940      1349
           1     0.5472    0.5044    0.5249      1356
           2     0.4606    0.4517    0.4561      1357
           3     0.6526    0.6902    0.6709      1301

    accuracy                         0.5875      5363
   macro avg     0.5851    0.5888    0.5865      5363
weighted avg     0.5842    0.5875    0.5855      5363

[[956 117 158 118]
 [174 684 377 121]
 [163 342 613 239]
 [113 107 183 898]]
Cross Validation Scores are [0.62136465 0.58724832 0.59116331 0.59899329 0.60234899 0.59116331
 0.60268607 0.59037493 0.60156687 0.58589815 0.59955257 0.60794183
 0.60067114 0.60682327 0.59284116 0.60794183 0.60492445 0.60492445
 0.6155568  0.60324566 0.59675615 0.5917226  0.59395973 0.60402685
 0.6090604  0.61521253 0.60996083 0.60156687 0.60660325 0.58533856]
Average Cross Validation score :0.6010479613514959


In [20]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier  # import random forest model
rfc = RandomForestClassifier(n_estimators=100)  # create the model with 100 trees
rfc.fit(x_train, y_train)  # train the model
predictions = rfc.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = rfc.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(rfc, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.8590    0.8310    0.8448      1349
           1     0.5981    0.7242    0.6551      1356
           2     0.6408    0.5114    0.5689      1357
           3     0.8305    0.8509    0.8405      1301

    accuracy                         0.7280      5363
   macro avg     0.7321    0.7294    0.7273      5363
weighted avg     0.7309    0.7280    0.7260      5363

[[1121  142   38   48]
 [  56  982  275   43]
 [  96  432  694  135]
 [  32   86   76 1107]]
Cross Validation Scores are [0.74888143 0.75279642 0.74832215 0.74440716 0.75727069 0.74888143
 0.75153889 0.75209849 0.74426413 0.7700056  0.73210291 0.76006711
 0.7466443  0.74720358 0.76510067 0.7746085  0.75657527 0.75321768
 0.75097929 0.74874091 0.74496644 0.74832215 0.74720358 0.76621924
 0.74944072 0.74272931 0.75265809 0.75601567 0.75209849 0.74874091]
Average Cross Validation score :0.7520700397727059


In [21]:
# K- nearest neighbor
from sklearn.neighbors import KNeighborsClassifier  # import KNN model
KN = KNeighborsClassifier()  # create the model
KN.fit(x_train, y_train)  # train the model
predictions = KN.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = KN.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(KN, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.6663    0.9548    0.7849      1349
           1     0.5780    0.2677    0.3659      1356
           2     0.6597    0.5372    0.5922      1357
           3     0.7171    0.9354    0.8119      1301

    accuracy                         0.6707      5363
   macro avg     0.6553    0.6738    0.6387      5363
weighted avg     0.6547    0.6707    0.6367      5363

[[1288   26   28    7]
 [ 439  363  302  252]
 [ 191  216  729  221]
 [  15   23   46 1217]]
Cross Validation Scores are [0.70917226 0.71085011 0.70357942 0.69742729 0.70693512 0.70581655
 0.70005596 0.70900951 0.70397314 0.72299944 0.69854586 0.73378076
 0.70525727 0.7041387  0.71756152 0.71588367 0.71460548 0.69781757
 0.69781757 0.68998321 0.69519016 0.71029083 0.70637584 0.70749441
 0.70022371 0.7041387  0.70677112 0.71012871 0.71348629 0.71460548]
Average Cross Validation score :0.7071305229958516


In [22]:
# GaussianNB
from sklearn.naive_bayes import GaussianNB  # import Gaussian Naive Bayes model
NB = GaussianNB()  # create the model
NB.fit(x_train, y_train)  # train the model
predictions = NB.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = NB.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(NB, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.4321    0.6294    0.5124      1349
           1     0.3699    0.2087    0.2669      1356
           2     0.3890    0.2181    0.2795      1357
           3     0.4487    0.6457    0.5295      1301

    accuracy                         0.4229      5363
   macro avg     0.4099    0.4255    0.3971      5363
weighted avg     0.4095    0.4229    0.3955      5363

[[849 206 111 183]
 [561 283 218 294]
 [329 177 296 555]
 [226  99 136 840]]
Cross Validation Scores are [0.41778523 0.41498881 0.41442953 0.42673378 0.43568233 0.42281879
 0.41801903 0.41130386 0.4040291  0.43256855 0.41666667 0.42281879
 0.41163311 0.41219239 0.43736018 0.42281879 0.42753218 0.41410185
 0.40514829 0.42865137 0.4155481  0.41331096 0.43008949 0.41834452
 0.42114094 0.41778523 0.44040291 0.41130386 0.42137661 0.41410185]
Average Cross Validation score :0.42002290342005205


In [23]:
# support vector machine
from sklearn.svm import LinearSVC  # import linear SVM model
classifier = LinearSVC()  # create the model
classifier.fit(x_train, y_train)  # train the model
y_predict = classifier.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, y_predict, digits=4))  # show test metrics
print(confusion_matrix(y_test, y_predict))  # show test confusion matrix
y_score = classifier.fit(x_train, y_train).decision_function(x_test)  # get decision scores

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, y_predict)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(classifier, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.4247    0.6983    0.5282      1349
           1     0.3475    0.1025    0.1583      1356
           2     0.3819    0.1584    0.2240      1357
           3     0.4056    0.6802    0.5082      1301

    accuracy                         0.4067      5363
   macro avg     0.3899    0.4099    0.3547      5363
weighted avg     0.3897    0.4067    0.3528      5363

[[942  93  77 237]
 [646 139 163 408]
 [373 117 215 652]
 [257  51 108 885]]
Cross Validation Scores are [0.42281879 0.39653244 0.39485459 0.41107383 0.42561521 0.40324385
 0.41354225 0.41074426 0.39003917 0.43368774 0.39765101 0.40380313
 0.40939597 0.40100671 0.42337808 0.41275168 0.41186346 0.40067152
 0.4034695  0.42977057 0.40380313 0.41442953 0.40883669 0.42058166
 0.4155481  0.40212528 0.42081701 0.40906547 0.41578064 0.39675434]
Average Cross Validation score :0.4101218532052896


In [24]:
# AdaBoostClassifier
from sklearn.ensemble import AdaBoostClassifier  # import AdaBoost model
abc = AdaBoostClassifier(n_estimators=50, learning_rate=1)  # create the model
model = abc.fit(x_train, y_train)  # train the model
y_pred = model.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, y_pred, digits=4))  # show test metrics
print(confusion_matrix(y_test, y_pred))  # show test confusion matrix
y_score = abc.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(abc, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.4888    0.6464    0.5567      1349
           1     0.4805    0.2456    0.3250      1356
           2     0.3382    0.3721    0.3544      1357
           3     0.4594    0.4919    0.4751      1301

    accuracy                         0.4382      5363
   macro avg     0.4417    0.4390    0.4278      5363
weighted avg     0.4415    0.4382    0.4271      5363

[[872 136 156 185]
 [428 333 416 179]
 [295 168 505 389]
 [189  56 416 640]]
Cross Validation Scores are [0.46756152 0.46420582 0.46700224 0.46923937 0.4647651  0.46252796
 0.46334639 0.48461108 0.45439284 0.46446558 0.43847875 0.47651007
 0.46532438 0.43903803 0.47203579 0.46588367 0.4812535  0.45103525
 0.44432009 0.47341914 0.45693512 0.46252796 0.47091723 0.45525727
 0.46196868 0.46029083 0.46502518 0.47789591 0.46950196 0.45775042]
Average Cross Validation score :0.46358290487225035


In [25]:
# GradientBoostingClassifier
from sklearn.ensemble import GradientBoostingClassifier  # import gradient boosting model
GB = GradientBoostingClassifier()  # create the model
model = GB.fit(x_train, y_train)  # train the model
predictions = model.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = GB.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(GB, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.7192    0.6701    0.6938      1349
           1     0.5614    0.7522    0.6429      1356
           2     0.4697    0.2859    0.3555      1357
           3     0.6282    0.7064    0.6650      1301

    accuracy                         0.6025      5363
   macro avg     0.5946    0.6037    0.5893      5363
weighted avg     0.5941    0.6025    0.5883      5363

[[ 904  193   87  165]
 [  56 1020  228   52]
 [ 153  489  388  327]
 [ 144  115  123  919]]
Cross Validation Scores are [0.60794183 0.61577181 0.598434   0.60458613 0.61689038 0.60346756
 0.61331841 0.61108002 0.59932848 0.61331841 0.58836689 0.61017897
 0.58892617 0.60961969 0.61912752 0.61241611 0.61779519 0.59932848
 0.60660325 0.62339116 0.58780761 0.60458613 0.59675615 0.64038031
 0.62024609 0.6163311  0.63122552 0.60268607 0.60772244 0.5965305 ]
Average Cross Validation score :0.6088054125265454


In [26]:
# XGBoost
from numpy import loadtxt  # import loadtxt from NumPy
from xgboost import XGBClassifier  # import XGBoost model
XGB = XGBClassifier()  # create the model
model = XGB.fit(x_train, y_train)  # train the model
predictions = model.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = XGB.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(XGB, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.8262    0.7821    0.8035      1349
           1     0.5679    0.7308    0.6391      1356
           2     0.5840    0.4355    0.4989      1357
           3     0.7795    0.7963    0.7878      1301

    accuracy                         0.6849      5363
   macro avg     0.6894    0.6862    0.6824      5363
weighted avg     0.6883    0.6849    0.6811      5363

[[1055  182   46   66]
 [  70  991  266   29]
 [ 109  459  591  198]
 [  43  113  109 1036]]
Cross Validation Scores are [0.7114094  0.70861298 0.70246085 0.68680089 0.72483221 0.69686801
 0.69557918 0.70677112 0.70900951 0.71572468 0.700783   0.71252796
 0.69239374 0.69574944 0.71644295 0.70637584 0.69949636 0.70789032
 0.70229435 0.70789032 0.68176734 0.70525727 0.69742729 0.71700224
 0.70749441 0.69463087 0.70677112 0.70453274 0.70005596 0.69781757]
Average Cross Validation score :0.7037556643035059


In [27]:
# Bagging
from sklearn.ensemble import BaggingClassifier
BC=BaggingClassifier()
model=BC.fit(x_train, y_train)
predictions = model.predict(x_test)
from sklearn.metrics import classification_report,confusion_matrix
print(classification_report(y_test,predictions,digits=4))
print(confusion_matrix(y_test,predictions))
y_score = BC.fit(x_train, y_train).predict_proba(x_test)

from sklearn.metrics import cohen_kappa_score
cohen_kappa_score(y_test,predictions)

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import  cross_val_score,KFold
kf = RepeatedStratifiedKFold(n_splits = 10, n_repeats=3, random_state=1)
scores = cross_val_score(BC, x, y, cv = kf)
print("Cross Validation Scores are {}".format(scores))
print("Average Cross Validation score :{}".format(scores.mean()))

              precision    recall  f1-score   support

           0     0.7539    0.7813    0.7674      1349
           1     0.5558    0.6431    0.5962      1356
           2     0.5365    0.4333    0.4794      1357
           3     0.7615    0.7610    0.7612      1301

    accuracy                         0.6534      5363
   macro avg     0.6519    0.6547    0.6511      5363
weighted avg     0.6507    0.6534    0.6498      5363

[[1054  145   79   71]
 [ 120  872  298   66]
 [ 152  444  588  173]
 [  72  108  131  990]]
Cross Validation Scores are [0.66946309 0.68400447 0.67841163 0.65715884 0.68903803 0.67170022
 0.67431449 0.66983772 0.66983772 0.68047006 0.67225951 0.68344519
 0.66442953 0.67002237 0.67281879 0.68232662 0.6737549  0.66983772
 0.67431449 0.66703973 0.66610738 0.66387025 0.68008949 0.68847875
 0.66722595 0.66946309 0.67879127 0.66536094 0.67879127 0.66032457]
Average Cross Validation score :0.67309960243986
